# Notebook 1 - Limpieza del dataset BUSI

**Deep Learning en Python para el Modelado Predictivo** - Trabajo academico

Este notebook hace **solo la limpieza de datos**:
1. Identifica las imagenes no validas segun los numeros del fichero *Observaciones a Eliminar*
   (tienen el cartel "Right/Left" o la zona del tumor senalada).
2. Muestra ejemplos de las imagenes eliminadas.
3. Copia las imagenes **buenas** a `Dataset_BUSI_clean` y las **descartadas** a
   `Dataset_BUSI_descartadas`, para poder contrastar las dos carpetas y verificar la limpieza.

Pensado para ejecutarse en un **servidor local** (sin Google Drive): trabaja con rutas
locales y crea las carpetas necesarias automaticamente.

## Instrucciones
1. Ajusta `INPUT_DIR` a la carpeta `Dataset_BUSI_with_GT` de tu servidor.
2. Ejecuta las celdas en orden. Las carpetas de salida se crean solas.
Es un proceso ligero (solo copia ficheros): no necesita GPU.

In [ ]:
# Montaje de Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# ----------------- CONFIGURACION (AJUSTA si tu carpeta esta en otra ruta) -----------------
BASE        = '/content/drive/MyDrive/Deep Learning'
INPUT_DIR   = BASE + '/Dataset_BUSI_with_GT'        # dataset ORIGINAL
OUTPUT_DIR  = BASE + '/Dataset_BUSI_clean'          # imagenes BUENAS (se crea)
DISCARD_DIR = BASE + '/Dataset_BUSI_descartadas'    # imagenes ELIMINADAS (se crea)
CLASSES     = ['benign', 'malignant', 'normal']
SEED        = 42

assert os.path.isdir(INPUT_DIR), 'No se encuentra INPUT_DIR: ' + INPUT_DIR
print('INPUT_DIR =', INPUT_DIR)
print('Contenido:', os.listdir(INPUT_DIR))

## 1. Imagenes a eliminar

El fichero *Observaciones a Eliminar* marca las imagenes con anotaciones del ecografo
(cartel "Right/Left" o zona del tumor senalada). Si no se eliminan, el clasificador
podria aprender esas marcas en vez del tejido (*data leakage*).

In [ ]:
# Numeros de imagen a eliminar (fichero "Observaciones a Eliminar")
# Escanear bien la observación 82 y 85,88,98 presentan peculiaridades de reloj y de negros
REMOVE = {
    'malignant': [2,10,11,12,13,14,15,26,53,61,64,65,67,69,70,76,78,80,81,83,95,98,101,102,106,108,109,112,113,110,111,114,117,118,119,121,122,123,124,125,126,127,128,
                  129,130,131,132,139,140,143,145,148,153,157,163,167,169,172,180,182,194,196,198,201,205,207],
    'benign':    [1,2,3,4,5,6,9,11,13,14,22,34,46,52,60,64,79,97,102,112,115,123,134,138,141,195,196,199,200,201,202,203,204,205,206,207,208,209,
                  210,211,212,
                  213,214,215,216,217,218,220,221,222,223,224,225,226,227,228,229,230,231,232,234,235,236,237,238,
                  239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,
                  263,264,265,266,267,268,270,271,273,274,275,276,277,278,279,280,281,282,
                  284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,300,301,302,303,304,305,306,309,316,318,319,320,321,322,323,328,330,331,334,343,345,359,362,366,367,
                  373,374,379,380,382,384,389,390,392,393,397,401,407,408,409,410,416,422,426,427,428,432,433,436],
    'normal':    [25,31,77,114,118,121,122],
}
for c in CLASSES:
    print(c, '->', len(REMOVE[c]), 'imagenes a eliminar')
print('TOTAL a eliminar:', sum(len(v) for v in REMOVE.values()))

In [ ]:
def numero_imagen(nombre):
    m = re.search(r'\((\d+)\)', nombre)
    return int(m.group(1)) if m else -1

def construir_tabla(data_dir):
    filas = []
    for cls in CLASSES:
        cdir = os.path.join(data_dir, cls)
        ficheros = sorted(os.listdir(cdir))
        imagenes = [f for f in ficheros if f.lower().endswith('.png') and '_mask' not in f.lower()]
        for f in imagenes:
            stem = f[:-4]
            mascaras = [m for m in ficheros if m.startswith(stem + '_mask')]
            num = numero_imagen(f)
            filas.append({'filepath': os.path.join(cdir, f), 'filename': f,
                          'class': cls, 'number': num, 'masks': mascaras,
                          'removed': num in REMOVE[cls]})
    return pd.DataFrame(filas)

df_all = construir_tabla(INPUT_DIR)
print('Total de imagenes encontradas:', len(df_all))
df_all.head()

In [ ]:
# Comprobacion + resumen
for cls in CLASSES:
    encontradas = set(df_all[df_all['class'] == cls]['number'])
    faltan = [n for n in REMOVE[cls] if n not in encontradas]
    if faltan:
        print('AVISO [' + cls + ']: no se encontraron las imagenes', faltan)

antes   = df_all['class'].value_counts()
despues = df_all[~df_all['removed']]['class'].value_counts()
resumen = pd.DataFrame({'antes': antes, 'eliminadas': antes - despues,
                        'despues': despues}).loc[CLASSES]
resumen.loc['TOTAL'] = resumen.sum()
print(resumen)

In [ ]:
# Visualizacion: imagenes por clase antes / eliminadas / despues
fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(CLASSES))
ancho = 0.27
b1 = ax.bar(x - ancho, resumen.loc[CLASSES, 'antes'].values,      ancho, label='Antes',      color='steelblue')
b2 = ax.bar(x,         resumen.loc[CLASSES, 'eliminadas'].values, ancho, label='Eliminadas', color='salmon')
b3 = ax.bar(x + ancho, resumen.loc[CLASSES, 'despues'].values,    ancho, label='Despues',    color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(CLASSES)
ax.set_ylabel('nº de imagenes')
ax.set_title('Imagenes por clase: antes / eliminadas / despues')
ax.legend()
for barras in [b1, b2, b3]:
    for b in barras:
        ax.text(b.get_x() + b.get_width()/2, b.get_height(), str(int(b.get_height())),
                ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.savefig('eliminadas_por_clase.png', dpi=120); plt.show()

# Porcentaje eliminado por clase
print('\nPorcentaje eliminado por clase:')
for cls in CLASSES:
    pct = 100.0 * resumen.loc[cls, 'eliminadas'] / max(int(resumen.loc[cls, 'antes']), 1)
    print('  %-10s: %5.1f%% eliminado  (%d de %d)' %
          (cls, pct, int(resumen.loc[cls, 'eliminadas']), int(resumen.loc[cls, 'antes'])))

# Comprobacion de balanceo final
counts_finales = resumen.loc[CLASSES, 'despues'].astype(int).values
ratio = counts_finales.max() / max(counts_finales.min(), 1)
print('\nRatio max/min tras la limpieza = %.2f' % ratio)
if ratio <= 1.5:
    print('-> Las clases han quedado APROXIMADAMENTE BALANCEADAS.')
else:
    print('-> El dataset sigue DESBALANCEADO -> el notebook 2 usara class_weight.')

In [ ]:
# Ejemplos de imagenes ELIMINADAS (figura para el minipaper)
n = min(8, int(df_all['removed'].sum()))
ej = df_all[df_all['removed']].sample(n=n, random_state=SEED)
plt.figure(figsize=(16, 8))
for i, (_, r) in enumerate(ej.iterrows()):
    plt.subplot(2, 4, i + 1)
    plt.imshow(Image.open(r['filepath']).convert('RGB'))
    plt.title(r['class'] + ' (' + str(r['number']) + ') - ELIMINADA', fontsize=9)
    plt.axis('off')
plt.suptitle('Ejemplos de imagenes eliminadas (carteles "Right/Left" / zona senalada)')
plt.tight_layout(); plt.show()

## 2. Separacion en dos carpetas

Cada imagen (con sus mascaras) se copia a una de dos carpetas nuevas:
- `Dataset_BUSI_clean` -> imagenes **validas** (las que usara el modelo).
- `Dataset_BUSI_descartadas` -> imagenes **eliminadas** (para revisarlas y comprobar
  que la limpieza es correcta).

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DISCARD_DIR, exist_ok=True)

for cls in CLASSES:
    os.makedirs(os.path.join(OUTPUT_DIR, cls), exist_ok=True)
    os.makedirs(os.path.join(DISCARD_DIR, cls), exist_ok=True)
    src_dir = os.path.join(INPUT_DIR, cls)
    sub = df_all[df_all['class'] == cls]
    for _, r in sub.iterrows():
        destino = DISCARD_DIR if r['removed'] else OUTPUT_DIR
        dst = os.path.join(destino, cls)
        shutil.copy2(r['filepath'], os.path.join(dst, r['filename']))
        for m in r['masks']:
            shutil.copy2(os.path.join(src_dir, m), os.path.join(dst, m))
    print(cls.ljust(10), '-> buenas:', int((~sub['removed']).sum()),
          '| descartadas:', int(sub['removed'].sum()))

print('\nImagenes buenas      ->', os.path.abspath(OUTPUT_DIR))
print('Imagenes descartadas ->', os.path.abspath(DISCARD_DIR))

In [ ]:
# Verificacion de las dos carpetas
def contar(carpeta):
    total = 0
    for cls in CLASSES:
        ficheros = os.listdir(os.path.join(carpeta, cls))
        imgs = [f for f in ficheros if '_mask' not in f.lower()]
        total += len(imgs)
        print('  ', cls.ljust(10), len(imgs), 'imagenes')
    print('   TOTAL:', total, '\n')
    return total

print('=== Dataset_BUSI_clean (buenas) ===')
n_buenas = contar(OUTPUT_DIR)
print('=== Dataset_BUSI_descartadas (eliminadas) ===')
n_malas = contar(DISCARD_DIR)
print('Comprobacion: buenas + descartadas =', n_buenas + n_malas,
      '(debe coincidir con el total original)')

## Limpieza completada

- `Dataset_BUSI_clean` -> imagenes validas (las usa el notebook 2).
- `Dataset_BUSI_descartadas` -> imagenes eliminadas (para revisarlas).

**Siguiente paso:** ejecuta `2_Entrenamiento_BUSI.ipynb`.